In [1]:
from pageindex import PageIndexClient
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import os

In [2]:
load_dotenv()  # Load environment variables from .env file

True

In [3]:
pageindex_client = PageIndexClient(api_key=os.getenv("PAGEINDEX_API_KEY"))
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [4]:
data_dir = "./data"
PDF_PATH = Path(data_dir) / "attention_is_all_you_need.pdf"

In [5]:
doc_id = "pi-cmnq7vdqx02op01pgdvnf1w07"

if Path.exists(Path(data_dir) / f"{doc_id}.json"):
    print(f"Document with doc_id {doc_id} already exists locally. Skipping upload.")
else:
    print("Uploading the PDF document to PageIndex...")
    result = pageindex_client.submit_document(PDF_PATH)

    doc_id = result["doc_id"]
    print(f"Document uploaded with doc_id: {doc_id}")

Document with doc_id pi-cmnq7vdqx02op01pgdvnf1w07 already exists locally. Skipping upload.


In [6]:
print("Building tree index... ")
import time

while True:
    status_result = pageindex_client.get_document(doc_id)
    status = status_result["status"]
    print(f"Current document status: {status}")

    if status == "completed":
        print("Document processing completed.")
        break
    elif status == "failed":
        print("Document processing failed.")
        exit(1)
    else:
        print("Document is still processing. Waiting for 10 seconds before checking again...")
        time.sleep(10)

Building tree index... 
Current document status: completed
Document processing completed.


In [9]:
import json

if not Path.exists(Path(data_dir) / f"{doc_id}.json"):
    print(f"Document with doc_id {doc_id} does not exist locally. Fetching from PageIndex...")
    tree_result = pageindex_client.get_tree(doc_id, node_summary=True)
    pageindex_tree = tree_result.get("result",[])

    print(f"Top-level sections: {len(pageindex_tree)}")
    print("Raw tree structure (first node):")
    print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

    # Optional: save locally
    import json
    with open(Path(data_dir) / "attention_is_all_you_need.json", "w") as f:
        json.dump(pageindex_tree, f)

In [10]:
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = "  " * indent + ("--" if indent > 0 else "")
        page = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

In [11]:
print_tree(pageindex_tree)

[0000] Attention Is All You Need (p.1)
  --[0001] Abstract (p.1)
  --[0002] 2 Background (p.2)
  --[0003] 3 Model Architecture (p.2)
    --[0004] 3.1 Encoder and Decoder Stacks (p.3)
    --[0005] 3.2 Attention (p.3)
      --[0006] 3.2.1 Scaled Dot-Product Attention (p.4)
      --[0007] 3.2.2 Multi-Head Attention (p.4)
      --[0008] 3.2.3 Applications of Attention in our Model (p.5)
    --[0009] 3.3 Position-wise Feed-Forward Networks (p.5)
    --[0010] 3.4 Embeddings and Softmax (p.5)
    --[0011] 3.5 Positional Encoding (p.6)
  --[0012] 4 Why Self-Attention (p.6)
  --[0013] 5 Training (p.7)
  --[0014] 6 Results (p.8)
    --[0015] 6.1 Machine Translation (p.8)
    --[0016] 6.2 Model Variations (p.8)
    --[0017] 6.3 English Constituency Parsing (p.9)
  --[0018] 7 Conclusion (p.10)
  --[0019] References (p.10)
  --[0020] Attention Visualizations (p.13)


In [12]:
def get_relevant_nodes(query: str, tree: list, model: str = "gpt-4o") -> dict:

    # Build simplified tree representation for LLM analysis
    def build_simplified_structure(node_list, depth=0):
        simplified = []
        for item in node_list:
            node_info = {
                "node_id": item["node_id"],
                "title": item["title"],
                "page_index": item.get("page_index", "?")
            }

            # Add text preview if available for better context
            content = item.get("text", "")
            if content:
                node_info["summary"] = content[:200]

            # Recursively process child nodes
            if "nodes" in item and item["nodes"]:
                node_info["children"] = build_simplified_structure(item["nodes"], depth + 1)

            simplified.append(node_info)

        return simplified

    # Create lightweight tree structure
    simplified_tree = build_simplified_structure(tree)

    # Construct analysis prompt
    analysis_prompt = f"""You are given a query and a hierarchical knowledge graph in JSON format.

Your task:
- Read node summaries
- Identify the most relevant nodes for the question
- Return only node_ids as a list

Question:
{query}

Knowledge Graph:
{json.dumps(simplified_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<Your step-by-step reasoning here>",
  "node_list": ["node_id_1", "node_id_2", ...]  // List of node_ids that are relevant
}}
"""

    # Get LLM analysis of relevant sections
    completion = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": analysis_prompt}],
        response_format={"type": "json_object"}
    )

    # Parse and return the structured response
    analysis_result = json.loads(completion.choices[0].message.content)
    return analysis_result

In [13]:
question = "How does the Transformer encode positional information, and what are the mathematical functions used?"

print(f"Running LLM Tree Search for query: '{question}'")
result = get_relevant_nodes(question, pageindex_tree)
print("LLM Tree Search Result:")
print(result.get("thinking", "N/A"))
print()
print(f"Relevant node IDs: {result.get('node_list', [])}")

Running LLM Tree Search for query: 'How does the Transformer encode positional information, and what are the mathematical functions used?'
LLM Tree Search Result:
The question is specifically about how the Transformer model encodes positional information and the mathematical functions used in this process. I need to identify nodes related to positional encoding. The hierarchical knowledge graph has a section titled '3 Model Architecture', which contains various aspects of the model. In particular, node '0011' is titled '3.5 Positional Encoding', which is very likely to contain information about how positional information is encoded mathematically in the Transformer architecture. This is the most relevant node in the context of the question.

Relevant node IDs: ['0011']


In [15]:
def find_nodes_by_ids(tree: list, target_ids:list) -> list:
    nodes = []
    for node in tree:
        if node["node_id"] in target_ids:
            nodes.append(node)
        if node.get("nodes"):
            nodes.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return nodes

In [16]:
def generate_answer_from_llm(question: str, nodes: list, model: str = "gpt-4o") -> str:
    # Handle edge case: no nodes retrieved
    if not nodes:
        return "No relevant sections found to answer the query."
    
    # Build context string from retrieved nodes
    section_contexts = []
    for node_data in nodes:
        section_header = f"[Section: '{node_data['title']}' | Page {node_data.get('page_index', '?')}]"
        section_content = node_data.get('text', 'Content not available')
        section_contexts.append(f"{section_header}\n{section_content}")
    
    # Combine all sections with clear delimiters
    full_context = "\n\n--\n\n".join(section_contexts)
    
    # Craft instruction prompt for answer generation
    system_instruction = f"""Answer the question using ONLY the context below.
Do not use any external knowledge.

Question:
{question}

Context:
{full_context}

Answer:
"""
    
    # Generate answer via LLM
    llm_response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": system_instruction}]
    )
    
    # Extract and clean the generated answer
    generated_answer = llm_response.choices[0].message.content.strip()
    return generated_answer

In [17]:
def vectorless_rag(query: str, tree: list) -> str:
    """
    Full end-to-end PageIndex RAG Pipeline:
    
    Step 1: LLM Tree search to find relevant node_ids
    Step 2: Find nodes in the tree matching those node_ids
    Step 3: Generate an answer grounded in the retrieved nodes, with citations
    """

    print("*" * 50)
    print(f"Query: {query}")
    print("*" * 50)

    search_result = get_relevant_nodes(query, tree)
    node_ids = search_result.get("node_list", [])

    print("LLM Tree Search Reasoning:")
    print(f"{search_result.get('thinking', '')[:200]}...")
    print(f"Relevant node IDs: {node_ids}")
    print("-" * 50)

    relevant_nodes = find_nodes_by_ids(tree, node_ids)

    print(f"Retrieved {len(relevant_nodes)} relevant nodes for answer generation.")
    for node in relevant_nodes:
        print(f"- {node['title']} (p.{node.get('page_index', '?')})")
    answer = generate_answer_from_llm(query, relevant_nodes)

    print("-" * 50)
    print("Generated Answer:")
    print(answer)
    return answer

In [18]:
vectorless_rag(question, pageindex_tree)

**************************************************
Query: How does the Transformer encode positional information, and what are the mathematical functions used?
**************************************************
LLM Tree Search Reasoning:
To answer the question about how the Transformer encodes positional information and the mathematical functions used, I need to find sections in the knowledge graph that describe the positional encodin...
Relevant node IDs: ['0011']
--------------------------------------------------
Retrieved 1 relevant nodes for answer generation.
- 3.5 Positional Encoding (p.6)
--------------------------------------------------
Generated Answer:
The Transformer encodes positional information using sine and cosine functions of different frequencies. The mathematical functions used are:

For even indices \(2i\):
\[ PE_{(pos, 2i)} = \sin(pos / 10000^{2i / d_{\text{model}}}) \]

For odd indices \(2i + 1\):
\[ PE_{(pos, 2i + 1)} = \cos(pos / 10000^{2i / d_{\text{model}}}) 

'The Transformer encodes positional information using sine and cosine functions of different frequencies. The mathematical functions used are:\n\nFor even indices \\(2i\\):\n\\[ PE_{(pos, 2i)} = \\sin(pos / 10000^{2i / d_{\\text{model}}}) \\]\n\nFor odd indices \\(2i + 1\\):\n\\[ PE_{(pos, 2i + 1)} = \\cos(pos / 10000^{2i / d_{\\text{model}}}) \\]\n\nEach dimension of the positional encoding corresponds to a sinusoid with wavelengths forming a geometric progression from \\(2\\pi\\) to \\(10000 \\cdot 2\\pi\\).'

In [19]:
question = "What does the term “self-attention” mean?"

In [20]:
vectorless_rag(question, pageindex_tree)

**************************************************
Query: What does the term “self-attention” mean?
**************************************************
LLM Tree Search Reasoning:
To determine the most relevant nodes for the query 'What does the term “self-attention” mean?', I need to identify sections in the knowledge graph that specifically mention 'self-attention'. Starting ...
Relevant node IDs: ['0004', '0012']
--------------------------------------------------
Retrieved 2 relevant nodes for answer generation.
- 3.1 Encoder and Decoder Stacks (p.3)
- 4 Why Self-Attention (p.6)
--------------------------------------------------
Generated Answer:
The term “self-attention” refers to a mechanism that connects all positions in a sequence with a constant number of sequentially executed operations. It allows for learning long-range dependencies by shortening the path length between any combination of positions in the input and output sequences, making it easier to capture such dependencies

'The term “self-attention” refers to a mechanism that connects all positions in a sequence with a constant number of sequentially executed operations. It allows for learning long-range dependencies by shortening the path length between any combination of positions in the input and output sequences, making it easier to capture such dependencies compared to recurrent layers. This mechanism is faster than recurrent layers when the sequence length is smaller than the representation dimensionality, which is often the case in state-of-the-art models for tasks like machine translation.'

In [21]:
question = "What are the two main parts of the Transformer model?"

In [22]:
vectorless_rag(question, pageindex_tree)

**************************************************
Query: What are the two main parts of the Transformer model?
**************************************************
LLM Tree Search Reasoning:
To identify the two main parts of the Transformer model, we need to find sections in the knowledge graph related to the model's architecture. We should look for summaries mentioning core components or...
Relevant node IDs: ['0003', '0004']
--------------------------------------------------
Retrieved 2 relevant nodes for answer generation.
- 3 Model Architecture (p.2)
- 3.1 Encoder and Decoder Stacks (p.3)
--------------------------------------------------
Generated Answer:
The two main parts of the Transformer model are the encoder and the decoder.


'The two main parts of the Transformer model are the encoder and the decoder.'